# 01.05 — Cardinality That Depends on Node Properties

> Builds on **01.03 — What Is Cardinality?**, which introduces constant cardinality.
> Read that first if you have not already.

In **01.03** cardinality was a single fixed bound: every `Movie` must have exactly
one `DIRECTED_BY` edge, regardless of anything else.  This works well when the
expectation is the same for every node of a type.

Sometimes it is not.  Consider a filmography database where `Person` nodes carry
a `kind` property — `"actor"`, `"director"`, or `"crew"` — and `Movie` nodes
carry a `genre` property — `"drama"` or `"blockbuster"`.  The domain has a clear
rule about `ACTED_IN` edges, but the rule **depends on both endpoints**:

| Person kind | Movie genre | Expected ACTED_IN count |
|-------------|-------------|-------------------------|
| `actor`     | `drama`     | 1 to 3 roles             |
| `actor`     | `blockbuster` | at most 1              |
| `director`  | *(any)*     | 0 — directors don't act |
| *(other)*   | *(any)*     | unrestricted (default)  |

A single `CardinalitySpec` cannot express this table.  This notebook shows how
`ConditionalCardinality` handles it, and what the validator does with it.

## The key idea: cardinality as a rule table

A `ConditionalCardinality` is a **lookup table** attached to one side of a
relationship type.  Each row is a pair of predicates — one matching the source
node's properties, one matching the target node's properties — together with a
`CardinalitySpec` that applies when both predicates match.

At validation time, for each node the validator:

1. Partitions the node's edges by the opposite endpoint's relevant property values.
2. For each partition, resolves which rule applies (most-specific match wins).
3. Checks the partition's count against the resolved spec.

A `default` spec is required.  It applies when no rule matches — preventing any
unrecognised combination from silently going unchecked.

Crucially, each rule is evaluated **per partition per node**, not globally.  An
actor who appeared in two dramas and one blockbuster is checked separately for
their drama edges (count 2, spec `1..3` — fine) and their blockbuster edge
(count 1, spec `0..1` — fine).

In [1]:
from orthograph.graph_definition.graph_definition import GraphDefinition
from orthograph.graph_definition.models import (
    CardinalitySpec,
    ConditionalCardinality,
    ConditionalRule,
    NodeModel,
    PropMatch,
    RelationshipModel,
)
from orthograph.graph_definition.validation import GraphValidator

## Schema validation comes first

Before any data is ever validated, `GraphDefinition` inspects the rule table
itself.  This happens at **model construction time**, not at data validation
time.  Two of the most important checks concern the discriminator properties —
the properties that the rules key on (`kind` and `genre` in our domain).

### Check 1: the discriminator property must exist on the node

If a rule discriminates on `kind` but the node model does not declare a `kind`
field, the model cannot be constructed.  There is nothing to read at data time.

In [2]:
from orthograph.diagnostics.result import GraphValidationError


# A Person model that has no 'kind' property at all.
class PersonWithoutKind(NodeModel):
    __label__ = "PersonWithoutKind"
    __uid_field__ = "name"
    name: str
    # 'kind' is simply absent.


class MovieForCheck(NodeModel):
    __label__ = "MovieForCheck"
    __uid_field__ = "title"
    title: str
    genre: str


card_referencing_missing_prop = ConditionalCardinality(
    rules=(
        ConditionalRule(
            source=PropMatch(
                {"kind": "actor"}
            ),  # 'kind' does not exist on PersonWithoutKind
            target=PropMatch({"genre": "drama"}),
            spec=CardinalitySpec(min=1, max=3),
        ),
    ),
    default="0..*",
)


class ActedInMissingProp(RelationshipModel):
    __label__ = "ACTED_IN_MISSING"
    __source_label__ = "PersonWithoutKind"
    __target_label__ = "MovieForCheck"
    __source_cardinality__ = card_referencing_missing_prop
    __target_cardinality__ = "0..*"
    role: str


try:
    GraphDefinition(
        name="MissingPropModel",
        node_types=[PersonWithoutKind, MovieForCheck],
        relationship_types=[ActedInMissingProp],
    )
except GraphValidationError as exc:
    for issue in exc.issues:
        print(f"[{issue.code}]")
        print(f"  {issue.message}")

[CARDINALITY_UNKNOWN_DISCRIMINATOR]
  ACTED_IN_MISSING source cardinality discriminates on PersonWithoutKind.kind, but 'kind' is not a declared property of PersonWithoutKind.
[CARDINALITY_DISCRIMINATOR_OPTIONAL]
  ACTED_IN_MISSING source cardinality discriminates on PersonWithoutKind.kind, but kind is optional (nullable); make it required or remove the rule.


### Check 2: the discriminator property must be required (non-nullable)

This is the subtler case.  Suppose `kind` *is* declared on the node, but as
`Optional[str]`.  The property exists — so check 1 passes — but at data time
some `Person` nodes may legally have `kind=None`.

An absent `kind` cannot match any rule.  It falls silently to `default`.  If
the author intended `default` to cover that, they should say so explicitly;
if they did not, they have a silent data-quality gap dressed as a schema.

Orthograph treats this as a **schema error**, not a data warning: a rule that
discriminates on an optional property is rejected at definition time.  The
fix is either to make the property required (`kind: str`) or to remove the
rule that references it.

In [3]:
from typing import Optional


# 'kind' exists but is nullable — some nodes may legitimately have kind=None.
class PersonWithOptionalKind(NodeModel):
    __label__ = "PersonWithOptionalKind"
    __uid_field__ = "name"
    name: str
    kind: Optional[str] = None  # nullable — rules cannot safely discriminate on this


card_referencing_optional_prop = ConditionalCardinality(
    rules=(
        ConditionalRule(
            source=PropMatch({"kind": "actor"}),  # 'kind' is declared but optional
            target=PropMatch({"genre": "drama"}),
            spec=CardinalitySpec(min=1, max=3),
        ),
    ),
    default="0..*",
)


class ActedInOptionalProp(RelationshipModel):
    __label__ = "ACTED_IN_OPTIONAL"
    __source_label__ = "PersonWithOptionalKind"
    __target_label__ = "MovieForCheck"
    __source_cardinality__ = card_referencing_optional_prop
    __target_cardinality__ = "0..*"
    role: str


try:
    GraphDefinition(
        name="OptionalPropModel",
        node_types=[PersonWithOptionalKind, MovieForCheck],
        relationship_types=[ActedInOptionalProp],
    )
except GraphValidationError as exc:
    for issue in exc.issues:
        print(f"[{issue.code}]")
        print(f"  {issue.message}")

[CARDINALITY_DISCRIMINATOR_OPTIONAL]
  ACTED_IN_OPTIONAL source cardinality discriminates on PersonWithOptionalKind.kind, but kind is optional (nullable); make it required or remove the rule.


### Why these are schema errors, not data warnings

Both checks fire at `GraphDefinition(...)`, before any data exists.  The
reasoning is the same in both cases: a rule that cannot be reliably evaluated
is not a rule — it is a latent silent pass.  The property being absent or
nullable does not change the expectation, it just makes it unverifiable.  That
is the exact failure mode this library exists to prevent, so it is rejected at
the declaration layer rather than deferred to data validation.

The practical implication: **fix the schema, not the data**.  If `kind` should
drive cardinality, declare it `kind: str` — required, non-nullable.  If `kind`
genuinely can be absent on some nodes, you need a different design (either a
second node label, or a constant cardinality that does not require reading `kind`).

## Declaring the domain — the correct model

With both discriminator properties declared as required, the model assembles
cleanly.

In [4]:
class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"
    name: str
    kind: str  # required: "actor" | "director" | "crew" | ...


class Movie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    genre: str  # required: "drama" | "blockbuster" | ...


# The rule table from the introduction, expressed in Python.
#
# Each ConditionalRule binds a (source, target) predicate pair to a spec.
# A PropMatch({prop: value}) matches nodes whose property equals value;
# PropMatch() (no conditions) is a wildcard that matches any value.
# Per ADR-032, source always describes the source-label node (Person) and
# target the target-label node (Movie).

acted_in_source_cardinality = ConditionalCardinality(
    rules=(
        ConditionalRule(
            source=PropMatch({"kind": "actor"}),
            target=PropMatch({"genre": "drama"}),
            spec="1..3",
        ),
        ConditionalRule(
            source=PropMatch({"kind": "actor"}),
            target=PropMatch({"genre": "blockbuster"}),
            spec="0..1",
        ),
        ConditionalRule(
            source=PropMatch({"kind": "director"}),
            target=PropMatch(),  # directors never act, any genre
            spec="0..0",
        ),
    ),
    default="0..*",  # crew, stunt, etc. — unrestricted
)


class ActedIn(RelationshipModel):
    __label__ = "ACTED_IN"
    __source_label__ = "Person"
    __target_label__ = "Movie"
    __source_cardinality__ = acted_in_source_cardinality
    __target_cardinality__ = "0..*"
    role: str


model = GraphDefinition(
    name="Filmography",
    node_types=[Person, Movie],
    relationship_types=[ActedIn],
)

print("Model assembled — definition-time rule checks passed.")
print(f"  Rules declared: {len(acted_in_source_cardinality.rules)}")
print(f"  Default:        {acted_in_source_cardinality.default}")

Model assembled — definition-time rule checks passed.
  Rules declared: 3
  Default:        min=0 max=None


## Scenario 1: valid data — multiple actors, multiple genres

An actor who appears in two dramas and one blockbuster satisfies all three
applicable rules simultaneously.  A director with no acting credits is also
fine — the `("director", "*")` rule says `ZERO`, and zero is within `0..0`.

In [5]:
nodes_valid = [
    {"__label__": "Person", "name": "Alice", "kind": "actor"},
    {"__label__": "Person", "name": "Nolan", "kind": "director"},
    {"__label__": "Movie", "title": "Drama A", "genre": "drama"},
    {"__label__": "Movie", "title": "Drama B", "genre": "drama"},
    {"__label__": "Movie", "title": "Blockbuster X", "genre": "blockbuster"},
]

# Alice: 2 drama roles (within 1..3) + 1 blockbuster role (within 0..1) — valid.
# Nolan: 0 acting credits — valid (ZERO means must not act, and he does not).
rels_valid = [
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Alice",
        "__target_uid__": "Drama A",
        "role": "Lead",
    },
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Alice",
        "__target_uid__": "Drama B",
        "role": "Supporting",
    },
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Alice",
        "__target_uid__": "Blockbuster X",
        "role": "Cameo",
    },
]

result = GraphValidator(model).validate(nodes_valid, rels_valid)
print(f"is_valid: {result.is_valid}")
print(f"issues:   {len(result.issues)}")

is_valid: True
issues:   0


## Scenario 2: violation — an actor with too many drama roles

Alice appears in four dramas.  The `("actor", "drama")` rule allows at most 3.
The validator checks this partition — count 4, spec `1..3` — and reports a
violation naming the specific (source kind, target genre) pair.

In [6]:
nodes_excess = [
    {"__label__": "Person", "name": "Alice", "kind": "actor"},
    {"__label__": "Movie", "title": "Drama A", "genre": "drama"},
    {"__label__": "Movie", "title": "Drama B", "genre": "drama"},
    {"__label__": "Movie", "title": "Drama C", "genre": "drama"},
    {"__label__": "Movie", "title": "Drama D", "genre": "drama"},
]

rels_excess = [
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Alice",
        "__target_uid__": "Drama A",
        "role": "Role A",
    },
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Alice",
        "__target_uid__": "Drama B",
        "role": "Role B",
    },
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Alice",
        "__target_uid__": "Drama C",
        "role": "Role C",
    },
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Alice",
        "__target_uid__": "Drama D",
        "role": "Role D",
    },
]

result = GraphValidator(model).validate(nodes_excess, rels_excess)
print(f"is_valid: {result.is_valid}")
for err in result.errors:
    print(f"  [{err.code}] {err.message}")
    print(
        f"    source_kind={err.context['source_kind']!r}  "
        f"target_genre={err.context['target_kind']!r}  "
        f"actual={err.context['actual']}  "
        f"expected={err.context['expected_min']}..{err.context['expected_max']}"
    )

is_valid: False
  [CARDINALITY_VIOLATION] Node 'Alice' (Person) has 4 outgoing ACTED_IN relationships for pair (source='actor', target=None), expected 1..3
    source_kind='actor'  target_genre=None  actual=4  expected=1..3


## Scenario 3: violation — a director who acted

The `("director", "*")` wildcard rule says `ZERO`: a director must have no
acting edges to *any* genre.  One acting credit is enough to trigger a violation.
The message names the partition as `(source='director', target='drama')`.

In [7]:
nodes_dir = [
    {"__label__": "Person", "name": "Nolan", "kind": "director"},
    {"__label__": "Movie", "title": "Drama A", "genre": "drama"},
]

rels_dir = [
    # Nolan sneaks into his own film as an actor.
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Nolan",
        "__target_uid__": "Drama A",
        "role": "Cameo",
    },
]

result = GraphValidator(model).validate(nodes_dir, rels_dir)
print(f"is_valid: {result.is_valid}")
for err in result.errors:
    print(f"  [{err.code}] {err.message}")

is_valid: False
  [CARDINALITY_VIOLATION] Node 'Nolan' (Person) has 1 outgoing ACTED_IN relationships for pair (source='director', target=None), expected 0..0


## Scenario 4: violation — a missing partition counted as zero

The `("actor", "drama")` rule has `min=1`: an actor who appears in the graph
but has **no drama edges** violates the rule even though there are no edges to
count.  The validator synthesises the missing partition with count 0 and checks
it against `min=1`.

This is the key difference from constant cardinality: it is not enough that the
partition simply does not appear in the data — the declared rule still fires.
The author who wrote `min=1` on the `(actor, drama)` rule is asserting: *every
actor must have taken at least one drama role.*

In [8]:
nodes_missing = [
    # An actor — but only in blockbusters, never in a drama.
    {"__label__": "Person", "name": "Alice", "kind": "actor"},
    {"__label__": "Movie", "title": "Blockbuster X", "genre": "blockbuster"},
    # A drama exists in the graph, so the validator knows the partition is possible.
    {"__label__": "Movie", "title": "Drama A", "genre": "drama"},
]

rels_missing = [
    # Alice acted in a blockbuster (fine: 0..1 — count 1 is within range).
    # Alice has NO drama edges — the (actor, drama) partition has count 0,
    # which violates min=1.
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Alice",
        "__target_uid__": "Blockbuster X",
        "role": "Lead",
    },
]

result = GraphValidator(model).validate(nodes_missing, rels_missing)
print(f"is_valid: {result.is_valid}")
for err in result.errors:
    print(f"  [{err.code}] {err.message}")
    print(
        f"    actual={err.context['actual']}  "
        f"(missing partition counts as 0, min=1 unmet)"
    )

is_valid: False
  [CARDINALITY_VIOLATION] Node 'Alice' (Person) has 0 outgoing ACTED_IN relationships for pair (source='actor', target=None), expected 1..3
    actual=0  (missing partition counts as 0, min=1 unmet)


## Scenario 5: unrecognised kind — the drift signal

A `Person` with `kind="crew"` matches no rule's source predicate (no rule has
`source={kind: "crew"}` and no source-wildcard rule exists).  The validator
applies `default="0..*"`, which admits any count — so the data is
**valid**.  But it also emits a `CARDINALITY_UNMATCHED_KIND` issue at severity
`INFO` to surface the fact that `crew` is not explicitly modelled.

This is intentional: the INFO is a **drift signal**, not an error.  It says
*"this kind exists in the data but has no rule — is that deliberate?"*  The
author can either add a rule for `crew` or confirm the default is sufficient.

In [9]:
nodes_crew = [
    {"__label__": "Person", "name": "Bob", "kind": "crew"},
    {"__label__": "Movie", "title": "Drama A", "genre": "drama"},
]

# Bob (crew) has one acting credit in a drama.
# 'crew' matches no rule — falls to default="0..*".
rels_crew = [
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Bob",
        "__target_uid__": "Drama A",
        "role": "Extra",
    },
]

result = GraphValidator(model).validate(nodes_crew, rels_crew)
print(f"is_valid: {result.is_valid}   (no errors — default permits any count)")
print()
info_issues = [i for i in result.issues if i.code == "CARDINALITY_UNMATCHED_KIND"]
print(f"INFO issues: {len(info_issues)}")
for info in info_issues:
    print(f"  [{info.severity.value.upper()}] {info.message}")

is_valid: True   (no errors — default permits any count)

INFO issues: 1
  [INFO] Node 'Bob' (Person) 'crew' matches no ACTED_IN source cardinality rule; the default bound applies.


## Scenario 6: the default floor

When an unrecognised kind is combined with a `default` that has `min > 0`, the
validator enforces that minimum against the node's **total** edge count on that
side.  This prevents a `min > 0` default from silently doing nothing for a
zero-edge node.

The example below declares `default="1..*"` — any unrecognised kind must
have at least one `ACTED_IN` edge.  An unrecognised `"stagehand"` with no edges
is a violation.

In [10]:
strict_source_card = ConditionalCardinality(
    rules=(
        ConditionalRule(
            source=PropMatch({"kind": "director"}),
            target=PropMatch(),
            spec="0..0",
        ),
    ),
    default="1..*",  # every unrecognised kind must have >=1 edge
)


class StrictActedIn(RelationshipModel):
    __label__ = "STRICT_ACTED_IN"
    __source_label__ = "Person"
    __target_label__ = "Movie"
    __source_cardinality__ = strict_source_card
    __target_cardinality__ = "0..*"
    role: str


strict_model = GraphDefinition(
    name="StrictFilmography",
    node_types=[Person, Movie],
    relationship_types=[StrictActedIn],
)

# A "stagehand" with no edges — matches no rule, falls to default="1..*".
# Total edge count is 0, which violates min=1.
nodes_stagehand = [
    {"__label__": "Person", "name": "Sam", "kind": "stagehand"},
]

result = GraphValidator(strict_model).validate(nodes_stagehand, [])
print(f"is_valid: {result.is_valid}   (default floor fires — min=1 unmet)")
for err in result.errors:
    print(f"  [{err.code}] {err.message}")
print()
for info in result.issues:
    if info.code == "CARDINALITY_UNMATCHED_KIND":
        print(f"  [{info.severity.value.upper()}] {info.message}")

is_valid: False   (default floor fires — min=1 unmet)
  [CARDINALITY_VIOLATION] Node 'Sam' (Person) 'stagehand' matches no STRICT_ACTED_IN source cardinality rule and has 0 outgoing relationships, violating the default bound 1..N

  [INFO] Node 'Sam' (Person) 'stagehand' matches no STRICT_ACTED_IN source cardinality rule; the default bound applies.


## How resolution works: most-specific rule wins

When multiple rules match a pair, the one with the highest **combined
specificity** (number of conditions on source + number on target) wins.  A
wildcard predicate has specificity 0; a predicate with one condition has
specificity 1.

In the table above, `("director", "*")` has combined specificity `1 + 0 = 1`.
If we added a narrower rule `("director", "drama")` with specificity `1 + 1 = 2`,
it would win over the wildcard for drama movies specifically.

Order of declaration is irrelevant — specificity determines the winner, not
position.  If two rules tie at the top specificity and can match the same pair,
the definition raises an error at construction time (`CARDINALITY_AMBIGUOUS_RULES`).

In [11]:
# Narrow rule overrides broad rule for the specific pair (director, drama).
# (director, blockbuster) still falls to the wildcard rule.

refined_card = ConditionalCardinality(
    rules=(
        ConditionalRule(
            source=PropMatch({"kind": "director"}),
            target=PropMatch(),  # broad: directors never act
            spec="0..0",
        ),
        ConditionalRule(
            source=PropMatch({"kind": "director"}),
            target=PropMatch(
                {"genre": "drama"}
            ),  # narrow: directors may cameo in a drama
            spec="0..1",
        ),
        ConditionalRule(
            source=PropMatch({"kind": "actor"}),
            target=PropMatch({"genre": "drama"}),
            spec="1..3",
        ),
        ConditionalRule(
            source=PropMatch({"kind": "actor"}),
            target=PropMatch({"genre": "blockbuster"}),
            spec="0..1",
        ),
    ),
    default="0..*",
)

# Resolve a few pairs by hand to confirm most-specific-wins.
test_pairs = [
    ({"kind": "director"}, {"genre": "drama"}, "expect 0..1 (narrow wins)"),
    ({"kind": "director"}, {"genre": "blockbuster"}, "expect 0..0 (broad applies)"),
    ({"kind": "actor"}, {"genre": "drama"}, "expect 1..3"),
    ({"kind": "crew"}, {"genre": "drama"}, "expect default 0..*"),
]

for src, tgt, note in test_pairs:
    spec = refined_card.resolve_for_pair(src, tgt)
    print(f"  ({src['kind']!r:10}, {tgt['genre']!r:14})  -> {spec.notation:6}   {note}")

  ('director', 'drama'       )  -> 0..1     expect 0..1 (narrow wins)
  ('director', 'blockbuster' )  -> 0..0     expect 0..0 (broad applies)
  ('actor'   , 'drama'       )  -> 1..3     expect 1..3
  ('crew'    , 'drama'       )  -> 0..*     expect default 0..*


## From a domain table to explicit `ConditionalRule` syntax

Before writing any code it helps to write down the domain rule as a table.
Each row becomes one `ConditionalRule`; the row that covers everything else
becomes `default`.

Per ADR-032's **absolute convention**, `rule.source` always describes the
relationship's source-label node and `rule.target` the target-label node —
regardless of which side's cardinality is declared.

Mapping:

| Table column | Explicit construction |
|---|---|
| Source property name + value in a row | `source=PropMatch({prop: value})` |
| Target property name + value in a row | `target=PropMatch({prop: value})` |
| *(any)* / wildcard on a side | `PropMatch()` (no conditions) |
| *(other)* / everything else | `default=` |
| Count range | `spec=` notation string `"min..max"` |

The examples below walk through three domain tables in the movie space:

1. **`SEQUEL_OF`** — only the source matters (target is always `PropMatch()`).
2. **`AWARDED`** — only the target matters (source is always `PropMatch()`).
3. **`REVIEWED`** — both sides matter.

## Example A: rules keyed on source only — `SEQUEL_OF`

A `SEQUEL_OF` edge runs from a sequel `Movie` back to its predecessor.
The rule depends entirely on the source movie's `franchise` property:

| Source `franchise` | Target | Expected `SEQUEL_OF` count (source side) |
|---|---|---|
| `"yes"` | *(any)* | exactly 1 — every sequel must name its predecessor |
| `"no"` | *(any)* | 0..0 — stand-alone films must not have a predecessor |
| *(other)* | *(any)* | 0..1 — default: one or no predecessor |

The target side is always a wildcard `PropMatch()` — we do not care about the
predecessor's own `franchise` value.  This is the common case when a
relationship type has a meaningful classification on one endpoint and a
homogeneous (or irrelevant) other endpoint.

In [ ]:
# Both source and target are Movie, discriminating only on source.franchise.


class FilmNode(NodeModel):
    __label__ = "Film"
    __uid_field__ = "title"
    title: str
    franchise: str  # required: "yes" | "no" | ...


sequel_of_source = ConditionalCardinality(
    rules=(
        ConditionalRule(
            source=PropMatch({"franchise": "yes"}),
            target=PropMatch(),  # target value ignored — wildcard
            spec="1..1",  # franchise film must name exactly one predecessor
        ),
        ConditionalRule(
            source=PropMatch({"franchise": "no"}),
            target=PropMatch(),  # target value ignored — wildcard
            spec="0..0",  # stand-alone film must not be a sequel
        ),
    ),
    default="0..1",  # unclassified films: zero or one predecessor
)


class SequelOf(RelationshipModel):
    __label__ = "SEQUEL_OF"
    __source_label__ = "Film"
    __target_label__ = "Film"
    __source_cardinality__ = sequel_of_source
    # target side: any film may have any number of sequels pointing back at it
    __target_cardinality__ = "0..*"


sequel_model = GraphDefinition(
    name="SequelGraph",
    node_types=[FilmNode],
    relationship_types=[SequelOf],
)
print("Schema assembled.")

# Confirm resolution for each source kind (target property irrelevant).
for franchise_val, note in [
    ("yes", "franchise film — must have predecessor"),
    ("no", "stand-alone — must not have predecessor"),
    ("unknown", "unclassified — zero or one predecessor"),
]:
    spec = sequel_of_source.resolve_for_pair(
        {"franchise": franchise_val},
        {"franchise": "yes"},  # target value irrelevant
    )
    print(f"  franchise={franchise_val!r:10}  -> {spec.notation:6}  ({note})")

In [ ]:
# Valid: a franchise film has exactly one predecessor; a stand-alone has none.
nodes_sequel_ok = [
    {"__label__": "Film", "title": "Part I", "franchise": "no"},
    {"__label__": "Film", "title": "Part II", "franchise": "yes"},
    {"__label__": "Film", "title": "Part III", "franchise": "yes"},
]
rels_sequel_ok = [
    {"__label__": "SEQUEL_OF", "__source_uid__": "Part II", "__target_uid__": "Part I"},
    {
        "__label__": "SEQUEL_OF",
        "__source_uid__": "Part III",
        "__target_uid__": "Part II",
    },
]

result = GraphValidator(sequel_model).validate(nodes_sequel_ok, rels_sequel_ok)
print(f"valid data  — is_valid: {result.is_valid}  issues: {len(result.issues)}")

# Invalid: stand-alone film has a predecessor (violates 0..0),
# and a franchise film has no predecessor (violates 1..1 — missing partition).
nodes_sequel_bad = [
    {"__label__": "Film", "title": "Orphan", "franchise": "yes"},  # needs predecessor
    {"__label__": "Film", "title": "Loner", "franchise": "no"},  # must have none
    {"__label__": "Film", "title": "Old Film", "franchise": "no"},
]
rels_sequel_bad = [
    # Loner (stand-alone) wrongly claims Old Film as its predecessor.
    {"__label__": "SEQUEL_OF", "__source_uid__": "Loner", "__target_uid__": "Old Film"},
]

result = GraphValidator(sequel_model).validate(nodes_sequel_bad, rels_sequel_bad)
print(f"invalid data — is_valid: {result.is_valid}")
for err in result.errors:
    print(f"  [{err.code}] {err.message}")

## Example B: rules keyed on target only — `AWARDED`

An `AWARDED` edge runs from an `AwardBody` to a `Person`.  The rule depends
entirely on the *target* `Person`'s `category` property — the award body is
homogeneous and needs no further partitioning.

**Absolute convention for `__target_cardinality__` (ADR-032):** a
`ConditionalRule`'s `source` predicate always matches the relationship's
**source-label** node and its `target` predicate always matches the
**target-label** node — regardless of which side's cardinality is being
declared.  Here the cardinality lives on `__target_cardinality__`, but the
discriminator is `AwardPerson.category` (the target-label node), so it goes on
`target=PropMatch({"category": ...})`.  The `AwardBody` source-label node is not
discriminated, so `source=PropMatch()` (a wildcard).

| Target `category` (`rule.target`) | Source (`rule.source`) | Expected `AWARDED` count (target side) |
|---|---|---|
| `"actor"` | wildcard `PropMatch()` | exactly 1 — one acting award per person |
| `"director"` | wildcard `PropMatch()` | exactly 1 — one directing award per person |
| `"crew"` | wildcard `PropMatch()` | 0..0 — crew are not eligible |
| *(other)* | wildcard `PropMatch()` | 0..1 — `default=` |

In [ ]:
class AwardBody(NodeModel):
    __label__ = "AwardBody"
    __uid_field__ = "name"
    name: str


class AwardPerson(NodeModel):
    __label__ = "AwardPerson"
    __uid_field__ = "name"
    name: str
    category: str  # required: "actor" | "director" | "crew" | ...


# awarded_target is on __target_cardinality__.
# Per ADR-032's absolute convention, rule.source always describes the
# source-label node (AwardBody) and rule.target the target-label node
# (AwardPerson) — regardless of which side the cardinality is declared on.
# The discriminator is AwardPerson.category, so it lives on rule.target;
# AwardBody is not discriminated, so rule.source is a wildcard PropMatch().
awarded_target = ConditionalCardinality(
    rules=(
        ConditionalRule(
            source=PropMatch(),  # AwardBody (source-label) — not discriminated
            target=PropMatch({"category": "actor"}),
            spec="1..1",  # actors receive exactly one award
        ),
        ConditionalRule(
            source=PropMatch(),
            target=PropMatch({"category": "director"}),
            spec="1..1",  # directors receive exactly one award
        ),
        ConditionalRule(
            source=PropMatch(),
            target=PropMatch({"category": "crew"}),
            spec="0..0",  # crew are not eligible
        ),
    ),
    default="0..1",
)


class Awarded(RelationshipModel):
    __label__ = "AWARDED"
    __source_label__ = "AwardBody"
    __target_label__ = "AwardPerson"
    __source_cardinality__ = "0..*"  # an award body may give many awards
    __target_cardinality__ = awarded_target


award_model = GraphDefinition(
    name="AwardGraph",
    node_types=[AwardBody, AwardPerson],
    relationship_types=[Awarded],
)
print("Schema assembled.")

# Valid: actor and director each get exactly one award; crew gets none.
nodes_award_ok = [
    {"__label__": "AwardBody", "name": "BAFTA"},
    {"__label__": "AwardPerson", "name": "Alice", "category": "actor"},
    {"__label__": "AwardPerson", "name": "Nolan", "category": "director"},
    {"__label__": "AwardPerson", "name": "Charlie", "category": "crew"},
]
rels_award_ok = [
    {"__label__": "AWARDED", "__source_uid__": "BAFTA", "__target_uid__": "Alice"},
    {"__label__": "AWARDED", "__source_uid__": "BAFTA", "__target_uid__": "Nolan"},
    # Charlie (crew) receives no award — correct.
]

result = GraphValidator(award_model).validate(nodes_award_ok, rels_award_ok)
print(f"valid data   — is_valid: {result.is_valid}  issues: {len(result.issues)}")

# Invalid: crew member receives an award (violates 0..0).
rels_award_bad = rels_award_ok + [
    {"__label__": "AWARDED", "__source_uid__": "BAFTA", "__target_uid__": "Charlie"},
]

result = GraphValidator(award_model).validate(nodes_award_ok, rels_award_bad)
print(f"invalid data — is_valid: {result.is_valid}")
for err in result.errors:
    print(f"  [{err.code}] {err.message}")

## Example C: rules keyed on both sides — `REVIEWED`

A `REVIEWED` edge runs from a `Person` to a `Movie`.  Both the reviewer's role
*and* the movie's genre drive the rule:

| Source `role` | Target `genre` | Expected `REVIEWED` count (source side) |
|---|---|---|
| `"critic"` | `"documentary"` | 1..* — critics must review every documentary |
| `"critic"` | *(any other genre)* | 0..* — critics may review non-docs freely |
| `"casual"` | *(any)* | 0..3 — casual viewers write at most three reviews |
| *(other)* | *(any)* | 0..0 — only critics and casual viewers write reviews |

Notice:
- Two critic rules — one narrow (`"documentary"`), one broad (`"*"`).  The
  narrow rule wins for documentaries (specificity 2 vs 1).
- The `"casual"` rule uses a target wildcard, so it applies to all genres.
- `default="0..0"` acts as a **prohibition**: any role not in the table must
  have zero reviews.  This is a useful pattern when the default should be
  *"this type of node must not participate at all"* rather than *"unrestricted"*.

In [ ]:
class Reviewer(NodeModel):
    __label__ = "Reviewer"
    __uid_field__ = "name"
    name: str
    role: str  # required: "critic" | "casual" | ...


class ReviewedMovie(NodeModel):
    __label__ = "ReviewedMovie"
    __uid_field__ = "title"
    title: str
    genre: str  # required: "documentary" | "drama" | ...


reviewed_source = ConditionalCardinality(
    rules=(
        ConditionalRule(
            source=PropMatch({"role": "critic"}),
            target=PropMatch({"genre": "documentary"}),
            spec="1..*",  # critics must review every documentary
        ),
        ConditionalRule(
            source=PropMatch({"role": "critic"}),
            target=PropMatch(),  # critics may review other genres freely
            spec="0..*",
        ),
        ConditionalRule(
            source=PropMatch({"role": "casual"}),
            target=PropMatch(),  # casual viewers: at most three reviews
            spec="0..3",
        ),
    ),
    default="0..0",  # any other role must not review at all
)


class Reviewed(RelationshipModel):
    __label__ = "REVIEWED"
    __source_label__ = "Reviewer"
    __target_label__ = "ReviewedMovie"
    __source_cardinality__ = reviewed_source
    __target_cardinality__ = "0..*"  # a movie may be reviewed any number of times


review_model = GraphDefinition(
    name="ReviewGraph",
    node_types=[Reviewer, ReviewedMovie],
    relationship_types=[Reviewed],
)

# Confirm the three distinct resolved specs.
print("Rule resolution:")
for role, genre, note in [
    ("critic", "documentary", "narrow critic rule wins"),
    ("critic", "drama", "broad critic rule applies"),
    ("casual", "documentary", "casual wildcard applies"),
    ("producer", "drama", "default prohibition"),
]:
    spec = reviewed_source.resolve_for_pair({"role": role}, {"genre": genre})
    print(f"  ({role!r:10}, {genre!r:14})  -> {spec.notation:6}  ({note})")

In [ ]:
# Valid: critic reviewed one documentary and one drama; casual reviewed two dramas.
# A producer exists but wrote no reviews — matches default 0..0, which is satisfied.
nodes_review_ok = [
    {"__label__": "Reviewer", "name": "Eve", "role": "critic"},
    {"__label__": "Reviewer", "name": "Frank", "role": "casual"},
    {"__label__": "Reviewer", "name": "Grace", "role": "producer"},
    {"__label__": "ReviewedMovie", "title": "Doc A", "genre": "documentary"},
    {"__label__": "ReviewedMovie", "title": "Drama B", "genre": "drama"},
]
rels_review_ok = [
    {"__label__": "REVIEWED", "__source_uid__": "Eve", "__target_uid__": "Doc A"},
    {"__label__": "REVIEWED", "__source_uid__": "Eve", "__target_uid__": "Drama B"},
    {"__label__": "REVIEWED", "__source_uid__": "Frank", "__target_uid__": "Drama B"},
    {"__label__": "REVIEWED", "__source_uid__": "Frank", "__target_uid__": "Doc A"},
]

result = GraphValidator(review_model).validate(nodes_review_ok, rels_review_ok)
print(f"valid data   — is_valid: {result.is_valid}  issues: {len(result.issues)}")

# Three violations at once:
# 1. Eve (critic) has no documentary review — missing partition violates 1..*.
# 2. Frank (casual) has four drama reviews — violates 0..3.
# 3. Grace (producer) wrote a review — violates the default 0..0.
nodes_review_bad = nodes_review_ok + [
    {"__label__": "ReviewedMovie", "title": "Drama C", "genre": "drama"},
    {"__label__": "ReviewedMovie", "title": "Drama D", "genre": "drama"},
]
rels_review_bad = [
    # Eve has no documentary review at all (missing partition, min=1 unmet).
    {"__label__": "REVIEWED", "__source_uid__": "Eve", "__target_uid__": "Drama B"},
    # Frank reviews four dramas (0..3 violated).
    {"__label__": "REVIEWED", "__source_uid__": "Frank", "__target_uid__": "Drama B"},
    {"__label__": "REVIEWED", "__source_uid__": "Frank", "__target_uid__": "Doc A"},
    {"__label__": "REVIEWED", "__source_uid__": "Frank", "__target_uid__": "Drama C"},
    {"__label__": "REVIEWED", "__source_uid__": "Frank", "__target_uid__": "Drama D"},
    # Grace (producer) writes a review — violates default 0..0.
    {"__label__": "REVIEWED", "__source_uid__": "Grace", "__target_uid__": "Drama B"},
]

result = GraphValidator(review_model).validate(nodes_review_bad, rels_review_bad)
print(f"invalid data — is_valid: {result.is_valid}")
for err in result.errors:
    print(f"  [{err.code}] {err.message}")

## Other definition-time rule-set checks

Two further checks guard the rule table's internal consistency.
Both fire at `GraphDefinition(...)` construction time.

**Duplicate rule** — two rules with the same `(source, target)` predicate
pair are meaningless; the second can never win.  Orthograph rejects this
rather than silently ignoring the duplicate.

**Ambiguous overlap** — two rules of equal specificity that can match the
same pair produce an undefined winner.  For example, `("actor", "*")` and
`("*", "drama")` both have specificity 1 and can both match
`(actor, drama)`.  Orthograph rejects this at definition time rather than
picking arbitrarily at validation time.

In [12]:
# --- Ambiguous overlap: two rules of equal specificity can co-match ---

try:
    ConditionalCardinality(
        rules=(
            ConditionalRule(
                source=PropMatch({"kind": "actor"}),
                target=PropMatch(),
                spec="1..3",  # specificity 1+0 = 1
            ),
            ConditionalRule(
                source=PropMatch(),
                target=PropMatch({"genre": "drama"}),
                spec="0..1",  # specificity 0+1 = 1
            ),
            # Both match (actor, drama) with equal specificity — ambiguous.
        ),
        default="0..*",
    )

    # The object is built fine — the check runs at GraphDefinition time.
    class AmbiguousActedIn(RelationshipModel):
        __label__ = "AMBIGUOUS_ACTED_IN"
        __source_label__ = "Person"
        __target_label__ = "Movie"
        __source_cardinality__ = ConditionalCardinality(
            rules=(
                ConditionalRule(
                    source=PropMatch({"kind": "actor"}),
                    target=PropMatch(),
                    spec="1..3",
                ),
                ConditionalRule(
                    source=PropMatch(),
                    target=PropMatch({"genre": "drama"}),
                    spec="0..1",
                ),
            ),
            default="0..*",
        )
        __target_cardinality__ = "0..*"
        role: str

    GraphDefinition(
        name="AmbiguousModel",
        node_types=[Person, Movie],
        relationship_types=[AmbiguousActedIn],
    )
except GraphValidationError as exc:
    for issue in exc.issues:
        print(f"[{issue.code}]")
        print(f"  {issue.message}")

[CARDINALITY_AMBIGUOUS_RULES]
  AMBIGUOUS_ACTED_IN source cardinality has ambiguous rules of equal specificity that can co-match: source={'kind': 'actor'}/target={} vs source={}/target={'genre': 'drama'}.


## Summary

`ConditionalCardinality` extends constant cardinality in one direction: the
bound is no longer a fixed number but a **lookup table keyed by endpoint
properties**.  The rest of the validation machinery is unchanged.

Key points:

- **Schema errors come first.**  A rule that discriminates on a property that
  is absent from the node model, or that is declared `Optional`, is rejected
  at `GraphDefinition(...)` construction time — before any data is touched.
  Fix the schema (`kind: str`, not `Optional[str]`), not the data.
- **`ConditionalCardinality(rules=(ConditionalRule(...), ...), default=...)`**
  is the authoring API.  Each `ConditionalRule` binds a
  `source=PropMatch({...})` / `target=PropMatch({...})` predicate pair to a
  `spec`.  Use `PropMatch()` (no conditions) for a wildcard on either side.
  Per ADR-032, `source` always matches the source-label node and `target` the
  target-label node, whichever side the cardinality is declared on.
- **Partitioning** is per-node: each node's edges are grouped by the opposite
  endpoint's property value, and each group is checked independently.
- **Missing partitions count as 0**: a rule with `min > 0` fires even if the
  relevant edges are simply absent.
- **`default` is required** and enforced: an unrecognised kind with `min > 0`
  in the default is a violation, not a silent pass.
- **Unrecognised kinds emit `CARDINALITY_UNMATCHED_KIND` (INFO)** regardless
  of whether the default fires an error, so model gaps surface as drift.
- **Ambiguous or duplicate rules** are also rejected at construction time —
  the rule table must be deterministic before any data is validated.

---

**What is not covered here (planned for later notebooks):**

- YAML syntax for `ConditionalCardinality` — see 02.01 once E40.6 is implemented.
- Live-database profiling of per-pair statistics — see 05.01 once E41 is implemented.
- Comparison: `compare(definition, profile)` reports conditional sides as
  `CARDINALITY_UNVERIFIABLE` until E41 delivers per-pair observed statistics.